# 06 Export dashboard data
Combines the results of notebooks 02-05 into **one small file** for the Streamlit dashboard (`dashboard/data/lsoa.parquet`), with latitude and longitude for mapping.
Contains only derived, aggregated open data (no raw files), so it can be kept in the repository.

In [10]:
from pathlib import Path
import pandas as pd
from pyproj import Transformer
CLEAN = Path("../data/clean"); OUT = Path("../dashboard/data"); OUT.mkdir(parents=True, exist_ok=True)
a = pd.read_parquet(CLEAN / "lsoa_allocation.parquet")
c = pd.read_parquet(CLEAN / "lsoa_clusters.parquet")[["LSOA21CD", "cluster"]]
df = a.merge(c, on="LSOA21CD", how="left")
t = Transformer.from_crs(27700, 4326, always_xy=True)
df["lon"], df["lat"] = t.transform(df["cent_x"].values, df["cent_y"].values)
df.shape

(33755, 44)

## Cluster names\nNames are read from the cluster profiles. **Check them** if you re-run clustering with different settings.

In [11]:
B = ["freq_morning", "freq_off_peak", "freq_evening"]
prof = df[df.cluster >= 0].groupby("cluster")[B].mean()
prof["level"] = prof[B].mean(axis=1)
names = {-1: "No service"}
for cl, r in prof.iterrows():
    shares = r[B] / r[B].sum()
    if shares["freq_off_peak"] > 0.55:      names[cl] = "Off-peak only"
    elif shares["freq_morning"] > 0.5:      names[cl] = "Morning-peak heavy"
    elif r["level"] >= 0.5 * prof["level"].max(): names[cl] = "High frequency, all day"
    else:                                    names[cl] = "Low frequency, all day"
df["cluster_name"] = df["cluster"].map(names)
pd.Series(names, name="name").to_frame().join(prof.round(1))

,name,freq_morning,freq_off_peak,freq_evening,level
-1,No service,NaN,NaN,NaN,NaN
0,"Low frequency, all day",8.9,9.4,8.7,9.0
1,"High frequency, all day",50.7,52.1,49.9,50.9
2,Off-peak only,1.8,4.6,0.8,2.4
3,Morning-peak heavy,3.9,2.2,1.2,2.4


## Save

In [12]:
keep = ["LSOA21CD", "LSOA21NM", "Urban_rura", "lat", "lon", "population", "pop_density", "imd_score", "imd_decile",
        "dist_nearest_stop_m", "n_stops", "freq_morning", "freq_off_peak", "freq_evening", "freq_mean", "freq_per_1000",
        "gap_score", "gap_rank", "demand", "x0", "x_opt", "unmet0", "unmet_opt", "cluster", "cluster_name",
        "region", "x0_morning", "xopt_morning", "x0_off_peak", "xopt_off_peak", "x0_evening", "xopt_evening"]
df[keep].round(5).to_parquet(OUT / "lsoa.parquet", index=False)
for f in ["allocation_sensitivity.csv", "allocation_bands.csv", "allocation_bands_sensitivity.csv", "allocation_regions.csv", "sensitivity.csv", "cluster_profiles.csv"]:
    pd.read_csv(CLEAN / f).to_csv(OUT / f, index=False)
print("saved", (OUT / "lsoa.parquet").stat().st_size / 1e6, "MB")

saved 4.201524 MB
